# Transformasi Pencerminan Titik

**Blok Utama** (biru): posisi asli titik GeoGebra A(2,3) B(2,4) C(3,4) D(3,3)  
**Blok Cermin** (merah): refleksi terhadap sumbu X

Transformasi:
$$T = \begin{bmatrix}1 & 0\\0 & s\end{bmatrix} \quad \text{dengan } s \text{ turun dari } 1 \to 0$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
from matplotlib.lines import Line2D
from IPython.display import HTML, display

In [ ]:
# ── Titik-titik dari GeoGebra kamu ──────────────────────────────
TITIK = {
    'A': (2, 3),  'B': (2, 4),  'C': (3, 4),  'D': (3, 3),
    'E': (2,-3),  'F': (3,-3),  'G': (2,-4),  'H': (3,-4),
    'I': (2, 2),  'J': (3, 2),  'K': (2, 1),  'L': (3, 1),
    'M': (2,-1),  'N': (3,-1),  'O': (2,-2),  'P': (3,-2),
}

FRAMES  = 160
PAUSE_F = 30

# ── Setup figure ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 8))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
ax.set_xlim(-6, 6)
ax.set_ylim(-6, 6)
ax.set_aspect('equal')
ax.axhline(0, color='black', linewidth=1.2, zorder=2)
ax.axvline(0, color='black', linewidth=1.2, zorder=2)
ax.grid(True, color='#cccccc', linewidth=0.5)
ax.tick_params(labelsize=9)
ax.set_title('Animasi Pencerminan Titik terhadap Sumbu X\n'
             '(Klik grafik untuk Pause/Play)',
             fontsize=11, pad=10)

# ── Warna per pasangan blok ──────────────────────────────────────
BLOK = [
    {'asli': ['A','B','C','D'], 'warna': '#1a6fb5'},
    {'asli': ['I','J','L','K'], 'warna': '#7c3aed'},
    {'asli': ['M','N','P','O'], 'warna': '#d97706'},
]

# artists per blok
blok_artists = []
for blok in BLOK:
    warna = blok['warna']
    pts   = [TITIK[k] for k in blok['asli']]
    xs    = [p[0] for p in pts]
    ys    = [p[1] for p in pts]

    # Kotak asli
    xmin, xmax = min(xs), max(xs)
    ymin, ymax = min(ys), max(ys)
    r_asli = patches.Rectangle(
        (xmin, ymin), xmax-xmin, ymax-ymin,
        lw=2, edgecolor=warna, facecolor='none', zorder=5
    )
    ax.add_patch(r_asli)
    d_asli, = ax.plot([], [], 'o', color=warna, ms=7, zorder=6)
    t_asli  = [ax.text(0,0, lbl, color=warna, fontsize=8,
                       fontweight='bold', zorder=7)
               for lbl in blok['asli']]

    # Kotak cermin
    r_cermin = patches.Rectangle(
        (xmin, -ymax), xmax-xmin, ymax-ymin,
        lw=2, edgecolor='#e74c3c', facecolor='none', zorder=5
    )
    ax.add_patch(r_cermin)
    d_cermin, = ax.plot([], [], 'o', color='#e74c3c', ms=7, zorder=6)
    lbl_cermin = [k+"'" for k in blok['asli']]
    t_cermin   = [ax.text(0,0, lbl, color='#e74c3c', fontsize=8,
                          fontweight='bold', zorder=7)
                  for lbl in lbl_cermin]

    blok_artists.append({
        'r_asli': r_asli, 'd_asli': d_asli, 't_asli': t_asli,
        'r_cermin': r_cermin, 'd_cermin': d_cermin, 't_cermin': t_cermin,
        'xmin': xmin, 'xmax': xmax, 'ymin': ymin, 'ymax': ymax
    })

# ── Info teks matriks ────────────────────────────────────────────
mat_txt = ax.text(-5.8, 5.5, '', fontsize=9, color='#222',
                  fontfamily='monospace',
                  bbox=dict(boxstyle='round,pad=0.4', facecolor='#f0f4ff',
                            edgecolor='#aaaaaa', alpha=0.9))

# ── Legend ───────────────────────────────────────────────────────
legend_elements = [
    Line2D([0],[0], color='#1a6fb5', lw=2, label='Blok Asli'),
    Line2D([0],[0], color='#e74c3c', lw=2, label="Blok Cermin (P')"),
]
ax.legend(handles=legend_elements, loc='upper right',
          fontsize=9, framealpha=0.9)

# ── State pause ──────────────────────────────────────────────────
state = {'paused': False}
def on_click(event):
    if event.inaxes == ax:
        state['paused'] = not state['paused']
fig.canvas.mpl_connect('button_press_event', on_click)

# ── Easing scale ─────────────────────────────────────────────────
def get_scale(frame):
    if frame < PAUSE_F:
        return 1.0
    if frame >= FRAMES - PAUSE_F:
        return 1.0
    mid = FRAMES // 2
    if frame <= mid:
        t = (frame - PAUSE_F) / (mid - PAUSE_F)
        t = 3*t**2 - 2*t**3
        return 1.0 - t
    else:
        t = (frame - mid) / (FRAMES - PAUSE_F - mid)
        t = 3*t**2 - 2*t**3
        return t

cur_frame = [0]

def animate(frame):
    if state['paused']:
        frame = cur_frame[0]
    else:
        cur_frame[0] = frame

    s = get_scale(frame)
    all_artists = [mat_txt]

    for ba in blok_artists:
        xmin = ba['xmin']; xmax = ba['xmax']
        ymin = ba['ymin']; ymax = ba['ymax']

        # Blok asli: y dikompres
        y0a = ymin * s
        y1a = ymax * s
        ba['r_asli'].set_y(y0a)
        ba['r_asli'].set_height(max(y1a - y0a, 1e-6))

        # sudut: BL, BR, TR, TL
        cx = [xmin, xmax, xmax, xmin]
        cy_a = [y0a, y0a, y1a, y1a]
        ba['d_asli'].set_data(cx, cy_a)
        off = [(-0.15,-0.2),(0.05,-0.2),(0.05,0.05),(-0.15,0.05)]
        for tx, px, py, (dx, dy) in zip(ba['t_asli'], cx, cy_a, off):
            tx.set_position((px+dx, py+dy))
        all_artists += [ba['d_asli'], ba['r_asli']] + ba['t_asli']

        # Blok cermin: refleksi
        y0c = -ymax * s
        y1c = -ymin * s
        ba['r_cermin'].set_y(y0c)
        ba['r_cermin'].set_height(max(y1c - y0c, 1e-6))
        cy_c = [y0c, y0c, y1c, y1c]
        ba['d_cermin'].set_data(cx, cy_c)
        off2 = [(-0.15,-0.25),(0.05,-0.25),(0.05,0.05),(-0.15,0.05)]
        for tx, px, py, (dx, dy) in zip(ba['t_cermin'], cx, cy_c, off2):
            tx.set_position((px+dx, py+dy))
        all_artists += [ba['d_cermin'], ba['r_cermin']] + ba['t_cermin']

    mat_txt.set_text(f'T = [1   0 ]\n     [0  {s:.2f}]\n\nScale Y: {s:.3f}')
    return all_artists

ani = animation.FuncAnimation(
    fig, animate, frames=FRAMES,
    interval=40, blit=True, repeat=True
)

plt.tight_layout()
display(HTML(ani.to_jshtml()))
plt.close()

## Tabel Koordinat

| Titik Asli | Cermin Sumbu X $P'(x,-y)$ | Cermin Sumbu Y $P''(-x,y)$ |
|:---:|:---:|:---:|
| $A(2,3)$ | $A'(2,-3)$ | $A''(-2,3)$ |
| $B(2,4)$ | $B'(2,-4)$ | $B''(-2,4)$ |
| $C(3,4)$ | $C'(3,-4)$ | $C''(-3,4)$ |
| $D(3,3)$ | $D'(3,-3)$ | $D''(-3,3)$ |
| $I(2,2)$ | $I'(2,-2)$ | $I''(-2,2)$ |
| $J(3,2)$ | $J'(3,-2)$ | $J''(-3,2)$ |
| $K(2,1)$ | $K'(2,-1)$ | $K''(-2,1)$ |
| $L(3,1)$ | $L'(3,-1)$ | $L''(-3,1)$ |
